# Explore correlation between Y and X 

## Incude data introduction (when was this survey conducted, combined from different year or states?)

X: Column 72-98, such as age, gender, education, Race/Ethnicity, and Metropolitan Status

Y: Political questions, such as "Thinking about important issues facing the country today, overall, would you say you tend to agree with Donald Trump on ..."


## Clean data:

* Treat **Refused / DK / No answer** as **missing (`NaN`)**
* **Do not** give them a numeric code 
* When exploring associations or fitting $f(Y\mid X)$, use **rows where that Y is observed**

In [2]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.utils import resample
from sklearn.metrics import confusion_matrix, accuracy_score, log_loss
from scipy.stats import chi2_contingency

In [3]:
DATA_PATH = "data/w61.csv"

X_COLS = [
    "F_METRO","F_CREGION","F_AGECAT","F_SEX","F_EDUCCAT","F_EDUCCAT2","F_HISP",
    "F_RACECMB","F_RACETHN","F_CITIZEN","F_NATIVITY2","F_MARITAL","F_RELIG",
    "F_BORN","F_ATTEND","F_RELIMP","F_PRAY","F_RELTRAD","F_INCOME","F_INCOME_RECODE",
    "F_PARTY_FINAL","F_PARTYLN_FINAL","F_PARTYSUM_FINAL","F_REG","F_IDEO","F_ACSWEB","F_VOLSUM"
]

NONRESP_TOKENS = {
    "Refused", "REFUSED",
    "Don't know", "Dont know", "Don't Know",
    "Don't know/Refused", "DK/REF", "DK",
    "Prefer not to answer", "No answer",
    "Not asked", "Skipped",
    "NA", "N/A", ""
}

def normalize_cat(s: pd.Series) -> pd.Series:
    s = s.astype("object")
    s = s.replace(list(NONRESP_TOKENS), np.nan)
    s = s.replace(r"^\s*$", np.nan, regex=True)
    return s

df = pd.read_csv(DATA_PATH, low_memory=False)

# --- Define Y columns by position rule from your w61.csv ---
cols = df.columns.tolist()
drop_prefix = cols[:5]          # columns 1–5
drop_last   = cols[-1:]         # last column
drop_set = set(drop_prefix + drop_last)

# Candidate block after removing 1–5 and last
candidate_cols = [c for c in cols if c not in drop_set]

# X columns present in df
X_in_df = [c for c in X_COLS if c in df.columns]

# Y = candidate minus X
Y_COLS = [c for c in candidate_cols if c not in set(X_in_df)]

print("Dropped (1–5):", drop_prefix)
print("Dropped (last):", drop_last[0])
print("n X:", len(X_in_df))
print("n Y:", len(Y_COLS))

# Normalize X and Y only
for c in X_in_df:
    df[c] = normalize_cat(df[c])
for c in Y_COLS:
    df[c] = normalize_cat(df[c])

# Missingness summaries (after token->NaN)
miss_y = df[Y_COLS].isna().mean().sort_values(ascending=False)
miss_x = df[X_in_df].isna().mean().sort_values(ascending=False)

# display(miss_y.head(10))
# display(miss_x.head(10))
print("n rows:", len(df))


Dropped (1–5): ['QKEY', 'INTERVIEW_START_W61', 'INTERVIEW_END_W61', 'LANG_W61', 'DEVICE_TYPE_W61']
Dropped (last): WEIGHT_W61
n X: 27
n Y: 66
n rows: 6395


## Decide Y

classify Y into different catagories. See the dependency of Y
Choose differet question type from TRUMPISSUE1_W61.


In [4]:
MIN_PAIR_N = 200
MAX_MISS_Y = 0.80
TOPK_X = 5

def cramers_v_bias_corrected(y: pd.Series, x: pd.Series) -> tuple[float, int]:
    m = y.notna() & x.notna()
    n = int(m.sum())
    if n == 0:
        return np.nan, 0

    tab = pd.crosstab(y[m], x[m])
    if tab.shape[0] < 2 or tab.shape[1] < 2:
        return np.nan, n

    chi2, _, _, _ = chi2_contingency(tab, correction=False)
    phi2 = chi2 / n
    r, k = tab.shape

    phi2corr = max(0.0, phi2 - ((k - 1) * (r - 1)) / max(n - 1, 1))
    rcorr = r - ((r - 1) ** 2) / max(n - 1, 1)
    kcorr = k - ((k - 1) ** 2) / max(n - 1, 1)
    denom = min(kcorr - 1, rcorr - 1)
    if denom <= 0:
        return np.nan, n

    return float(np.sqrt(phi2corr / denom)), n

n_total = len(df)
miss_y_map = miss_y.to_dict()

rows = []
for y in Y_COLS:
    if miss_y_map.get(y, 1.0) > MAX_MISS_Y:
        continue
    y_s = df[y]
    for x in X_in_df:
        v, n = cramers_v_bias_corrected(y_s, df[x])
        if (not np.isnan(v)) and (n >= MIN_PAIR_N):
            rows.append({"Y": y, "X": x, "v": v, "n_complete": n})

assoc = pd.DataFrame(rows)
# display(assoc.sort_values("v", ascending=False).head(30))

# Aggregate to per-Y score
g = assoc.groupby("Y")
summary = g.agg(
    mean_v=("v", "mean"),
    median_v=("v", "median"),
    max_v=("v", "max"),
    min_n=("n_complete", "min"),
    pairs=("v", "size")
).reset_index()

summary["missing_y"] = summary["Y"].map(miss_y_map).astype(float)
summary["coverage"] = np.sqrt(np.clip(summary["min_n"] / n_total, 0, 1))
summary["score"] = (0.6 * summary["mean_v"] + 0.4 * summary["max_v"]) * (1.0 - summary["missing_y"]) * summary["coverage"]

topx = (assoc.sort_values(["Y", "v"], ascending=[True, False])
             .groupby("Y").head(TOPK_X)
             .groupby("Y")
             .apply(lambda d: ", ".join(d["X"].tolist()))
             .rename("topX"))

summary = summary.merge(topx.reset_index(), on="Y", how="left")
summary_sorted = summary.sort_values("score", ascending=False)

display(summary_sorted.head(30))

# Optional save
summary_sorted.to_csv("Y_ranked_by_X_association.csv", index=False)
assoc.sort_values("v", ascending=False).to_csv("YX_pairwise_association.csv", index=False)


/tmp/ipykernel_164463/1818357879.py:61: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda d: ", ".join(d["X"].tolist()))


,Y,mean_v,median_v,max_v,min_n,pairs,missing_y,coverage,score,topX
54,TRUMPISSUE1_W61,0.167635,0.118212,0.697440,2330,27,0.006568,0.603612,0.227600,"F_PARTYLN_FINAL, F_PARTYSUM_FINAL, F_PARTY_FIN..."
42,TRUMPDESC_a_W61,0.163236,0.110389,0.689787,2322,27,0.009851,0.602575,0.223057,"F_PARTYLN_FINAL, F_PARTYSUM_FINAL, F_PARTY_FIN..."
45,TRUMPDESC_d_W61,0.157779,0.108663,0.669374,2322,27,0.011259,0.602575,0.215924,"F_PARTYLN_FINAL, F_PARTYSUM_FINAL, F_PARTY_FIN..."
48,TRUMPDESC_g_W61,0.155113,0.117151,0.629300,2319,27,0.009539,0.602185,0.205646,"F_PARTYLN_FINAL, F_PARTYSUM_FINAL, F_PARTY_FIN..."
43,TRUMPDESC_b_W61,0.154659,0.107350,0.631644,2317,27,0.012353,0.601926,0.205368,"F_PARTYLN_FINAL, F_PARTYSUM_FINAL, F_PARTY_FIN..."
55,TRUMPISSUE2_W61,0.160701,0.116741,0.616793,2284,27,0.026740,0.597624,0.199584,"F_PARTYLN_FINAL, F_PARTYSUM_FINAL, F_PARTY_FIN..."
57,WINLOSE_W61,0.156374,0.122298,0.580917,2272,27,0.025489,0.596052,0.189471,"F_PARTYSUM_FINAL, F_PARTY_FINAL, F_IDEO, F_PAR..."
51,TRUMPHELP_c_W61,0.159310,0.125623,0.532449,2301,27,0.021110,0.599844,0.181184,"F_PARTYLN_FINAL, F_PARTYSUM_FINAL, F_IDEO, F_P..."
44,TRUMPDESC_c_W61,0.139929,0.094105,0.542279,2317,27,0.011884,0.601926,0.178949,"F_PARTYLN_FINAL, F_PARTYSUM_FINAL, F_PARTY_FIN..."
47,TRUMPDESC_f_W61,0.124430,0.079916,0.558289,2305,27,0.018921,0.600365,0.175508,"F_PARTYLN_FINAL, F_PARTYSUM_FINAL, F_IDEO, F_P..."


## Columns in `summary_sorted`

### `v` 
bias-corrected Cramér’s V between categorical Y and categorical X (effect size in [0,1])


### `Y`

The name of the question column being ranked (e.g. `TRUMPISSUE1_W61`).

---

### `mean_v`

Average association strength of this $Y$ with all usable $X$’s:
$$
\mathrm{mean_v}(Y) = \frac{1}{m_Y}\sum_{j=1}^{m_Y} V(Y, X_j)
$$
where $m_Y$ = `pairs` (number of valid $X$’s for that $Y$).

---

### `median_v`

Median of the same set of effect sizes:
$$
{V(Y, X_j)}_{j=1}^{m_Y}
$$
More robust than `mean_v` if one $X$ is extremely strong.

---

### `max_v`

Strongest single association of that $Y$ with any $X$:
$$
\mathrm{max_v}(Y) = \max_{1\le j\le m_Y} V(Y, X_j)
$$
This is why `TRUMPISSUE1_W61` has large `max_v` (about 0.697): it has one $X$ (likely party ID) that strongly predicts it.

---

### `min_n`

Conservative usable sample size: the **smallest** `n_complete` among all valid $(Y,X)$ pairs for that $Y$:
$$
\mathrm{min_n}(Y)=\min_{1\le j\le m_Y} n_{\mathrm{complete}}(Y,X_j)
$$
Interpretation: “worst-case” complete-case $n$ across $X$’s.

---

### `pairs`

How many $X$ variables produced a valid $V$ and passed your filters for that $Y$:
$$
\mathrm{pairs}(Y)=m_Y
$$
In your screenshot it’s 27, which means all 27 $X$’s were usable (enough complete cases and at least 2 levels).

---

### `missing_y`

Missing proportion of $Y$ after converting nonresponse strings to NA:
$$
\mathrm{missing_y}(Y)=\frac{\left|{, i : Y_i \text{ is NA},}\right|}{n_{\mathrm{total}}}
$$
So `0.006568` means about 0.66% of rows have missing/nonresponse for that $Y$.

---

### `coverage`

A sample-size penalty derived from `min_n`:
$$
\mathrm{coverage}(Y) = \sqrt{\frac{\mathrm{min_n}(Y)}{n_{\mathrm{total}}}}
$$

* $n_{\mathrm{total}}$ is total rows in the dataset.
* The square root makes the penalty milder than linear.

If `coverage` is about 0.60, that implies:
$$
\frac{\mathrm{min_n}(Y)}{n_{\mathrm{total}}} \approx 0.60^2 \approx 0.36
$$

---

### `score`

The final ranking score used to sort $Y$’s:
$$
\mathrm{score}(Y)=\Big(0.6\cdot \mathrm{mean_v}(Y)+0.4\cdot \mathrm{max_v}(Y)\Big)\cdot \big(1-\mathrm{missing_y}(Y)\big)\cdot \mathrm{coverage}(Y)
$$

Interpretation:

* $0.6\cdot \mathrm{mean\_v}(Y)+0.4\cdot \mathrm{max\_v}(Y)$ rewards overall association but also credits one very strong driver.
* $1-\mathrm{missing\_y}(Y)$ penalizes $Y$’s with lots of missing/nonresponse.
* `coverage` penalizes $Y$’s that rely on a small complete-case subset for some $X$.



## TRUMPISSUE1_W61: Thinking about important issues facing the country today, overall, would you say you tend to agree with Donald Trump on ...
Answer options (coded 1–4):
* All or nearly all issues
* Many, but not all issues
* A few issues
* No or almost no issues

## Select X

In [ ]:
import pandas as pd

df = pd.read_csv(DATA_PATH, low_memory=False)

X_cols_72_98 = list(df.columns[71:98])  # 0-based slice
print("X (cols 72–98):")
for i, c in enumerate(X_cols_72_98, start=72):
    print(f"{i:>3}: {c}")

# Missingness of each X column (ranked high to low)
print("\n--- Missing % per X (high → low) ---")
for c in X_cols_72_98:
    df[c] = df[c].replace(list(NONRESP_TOKENS), np.nan)

miss_x = df[X_cols_72_98].isna().mean().sort_values(ascending=False)
for c, pct in miss_x.items():
    print(f"  {c:<25s} {pct*100:6.2f}%")


X (cols 72–98):
 F_METRO
 F_CREGION
 F_AGECAT
 F_SEX
 F_EDUCCAT
 F_EDUCCAT2
 F_HISP
 F_RACECMB
 F_RACETHN
 F_CITIZEN
 F_NATIVITY2
 F_MARITAL
 F_RELIG
 F_BORN
 F_ATTEND
 F_RELIMP
 F_PRAY
 F_RELTRAD
 F_INCOME
 F_INCOME_RECODE
 F_PARTY_FINAL
 F_PARTYLN_FINAL
 F_PARTYSUM_FINAL
 F_REG
 F_IDEO
 F_ACSWEB
 F_VOLSUM

--- Missing % per X (high → low) ---
  F_PARTYLN_FINAL            63.42%
  F_BORN                     38.22%
  F_REG                       5.11%
  F_INCOME                    4.10%
  F_INCOME_RECODE             4.10%
  F_IDEO                      2.11%
  F_RACECMB                   1.94%
  F_PARTY_FINAL               1.24%
  F_RACETHN                   1.19%
  F_RELTRAD                   0.47%
  F_RELIG                     0.47%
  F_HISP                      0.38%
  F_PRAY                      0.38%
  F_ATTEND                    0.38%
  F_AGECAT                    0.33%
  F_CITIZEN                   0.31%
  F_MARITAL                   0.31%
  F_NATIVITY2                 0.28%
  F_R


## Step 1 — Core political baseline (substantive inclusion)

* Politics: `F_PARTY_FINAL`, `F_PARTYSUM_FINAL`, `F_PARTYLN_FINAL`, `F_IDEO`, `F_REG`
* Demographics: `F_AGECAT`, `F_SEX`, `F_EDUCCAT`, `F_EDUCCAT2`, `F_RACECMB`, `F_RACETHN`, `F_HISP`, `F_INCOME`, `F_INCOME_RECODE`, `F_MARITAL`
* Context: `F_CREGION`, `F_METRO`
* Religion: `F_RELIG`, `F_ATTEND`, `F_RELIMP`, `F_PRAY`, `F_RELTRAD`, `F_BORN`
* Other: `F_CITIZEN`, `F_NATIVITY2`, `F_ACSWEB`, `F_VOLSUM`
* Race/Ethinicity combine into 1

## Step 2 — Apply a missingness threshold (preserve sample size)

Compute missingness of each candidate (X), and drop those above a threshold.

* `F_PARTYLN_FINAL` (~63%) → drop ## combine F_PARTYLN_FINAL F_PARTY_FINAL to create a new indicator with Democradic, lean demo...
* `F_BORN` (~40%) → drop in baseline ## go through questions in X, maybe keep 

* **Race** ⇒ keep `F_RACETHN`, drop `F_HISP` and `F_RACECMB`
* **Religion** ⇒ keep `F_RELIG`, `F_ATTEND`, `F_RELTRAD`, `F_BORN`
* **Citizenship/Nativity** ⇒ keep `F_CITIZEN`, `F_NATIVITY2`
* **Behavior** ⇒ keep `F_ACSWEB`, `F_VOLSUM`
* Keep the non-redundant recodes: **`F_EDUCCAT2`** (drop `F_EDUCCAT`), **`F_INCOME_RECODE`** (drop `F_INCOME`)
Keep F_INCOME drop recode
for education, keep the factor with more levels.

* Party block: you still need to pick **one** party variable for the fixed set; the recommended baseline is `F_PARTYSUM_FINAL` (and exclude `F_PARTY_FINAL`, `F_PARTYLN_FINAL`).

### Final fixed X set

X_fixed:  
    "F_METRO",
    "F_CREGION",
    "F_AGECAT",
    "F_SEX",
    "F_EDUCCAT2",
    "F_RACETHN",
    "F_CITIZEN",
    "F_NATIVITY2",
    "F_MARITAL",
    "F_RELIG",
    "F_BORN",
    "F_ATTEND",
    "F_RELTRAD",
    "F_INCOME_RECODE",
    "F_PARTYSUM_FINAL",
    "F_REG",
    "F_IDEO",
    "F_ACSWEB",
    "F_VOLSUM",


In [4]:
# Fill F_PARTY_FINAL and F_PARTYLN_FINAL from F_PARTYSUM_FINAL (row-wise, data-driven)
import numpy as np
import pandas as pd

PARTY_COL = "F_PARTY_FINAL"
LEAN_COL = "F_PARTYLN_FINAL"
SUM_COL = "F_PARTYSUM_FINAL"

df_party = pd.read_csv(DATA_PATH, low_memory=False).copy()

for c in [PARTY_COL, LEAN_COL, SUM_COL]:
    df_party[c] = normalize_cat(df_party[c])

# Build mapping from F_PARTYSUM_FINAL -> most common non-missing value in each target column
def mode_non_missing(s: pd.Series):
    s2 = s.dropna()
    if s2.empty:
        return np.nan
    m = s2.mode()
    return m.iloc[0] if not m.empty else np.nan

map_party = df_party.groupby(SUM_COL)[PARTY_COL].apply(mode_non_missing).to_dict()
map_lean = df_party.groupby(SUM_COL)[LEAN_COL].apply(mode_non_missing).to_dict()

before_party_na = int(df_party[PARTY_COL].isna().sum())
before_lean_na = int(df_party[LEAN_COL].isna().sum())

df_party[PARTY_COL] = df_party[PARTY_COL].where(
    df_party[PARTY_COL].notna(),
    df_party[SUM_COL].map(map_party)
 )
df_party[LEAN_COL] = df_party[LEAN_COL].where(
    df_party[LEAN_COL].notna(),
    df_party[SUM_COL].map(map_lean)
 )

after_party_na = int(df_party[PARTY_COL].isna().sum())
after_lean_na = int(df_party[LEAN_COL].isna().sum())

print("Missing counts before -> after")
print(f"  {PARTY_COL}: {before_party_na} -> {after_party_na}")
print(f"  {LEAN_COL}:  {before_lean_na} -> {after_lean_na}")

print("\nDerived mapping from F_PARTYSUM_FINAL:")
map_view = pd.DataFrame({
    SUM_COL: sorted(df_party[SUM_COL].dropna().unique()),
})
map_view[PARTY_COL] = map_view[SUM_COL].map(map_party)
map_view[LEAN_COL] = map_view[SUM_COL].map(map_lean)
display(map_view)

print("\nCheck filled results (first 20 rows):")
display(df_party[[SUM_COL, PARTY_COL, LEAN_COL]].head(20))

# Optional: if you want to use this filled version downstream, overwrite df or save
df = df_party.copy()
df_party.to_csv("data/w61_party_filled.csv", index=False)

Missing counts before -> after
  F_PARTY_FINAL: 79 -> 0
  F_PARTYLN_FINAL:  4056 -> 208

Derived mapping from F_PARTYSUM_FINAL:


,F_PARTYSUM_FINAL,F_PARTY_FINAL,F_PARTYLN_FINAL
0,DK/Refused/No lean,Something else,NaN
1,Dem/Lean Dem,Democrat,The Democratic Party
2,Rep/Lean Rep,Republican,The Republican Party



Check filled results (first 20 rows):


,F_PARTYSUM_FINAL,F_PARTY_FINAL,F_PARTYLN_FINAL
0,Rep/Lean Rep,Republican,The Republican Party
1,Dem/Lean Dem,Democrat,The Democratic Party
2,Rep/Lean Rep,Independent,The Republican Party
3,Rep/Lean Rep,Republican,The Republican Party
4,Rep/Lean Rep,Independent,The Republican Party
5,Rep/Lean Rep,Independent,The Republican Party
6,Rep/Lean Rep,Independent,The Republican Party
7,Dem/Lean Dem,Democrat,The Democratic Party
8,Dem/Lean Dem,Something else,The Democratic Party
9,Dem/Lean Dem,Independent,The Democratic Party


## Next Steps
* Re select X and Y: for X, be carefull to drop columns, maybe combine a few. For Y, choose different types of questions other than TRUMP
* Add distribution distance beside Accuarcy
* Future direciton: subsample and weight, make it unbiased

## Evaluation 

Given $N$ missing values, $K$ classes, true class index $y_i \in \{0,\ldots,K{-}1\}$, and predicted probability vector $\hat{p}_i \in \mathbb{R}^K$:

**Accuracy** : 
$$\text{Acc} = \frac{1}{N}\sum_{i=1}^{N} \mathbb{1}[\arg\max_k \hat{p}_{ik} = y_i]$$

**Log Loss** (cross-entropy) — penalizes confident wrong predictions:
$$\text{LogLoss} = -\frac{1}{N}\sum_{i=1}^{N} \ln(\hat{p}_{i, y_i})$$
Only looks at the probability assigned to the **true class**. 

**Brier Score** (multiclass) — MSE between predicted probabilities and the truth:
$$\text{Brier} = \frac{1}{N}\sum_{i=1}^{N}\sum_{k=0}^{K-1}(\mathbb{1}[y_i = k] - \hat{p}_{ik})^2$$
Uses the full probability vector, but treats all classes as unrelated (no notion of "closeness" between categories).

**Wasserstein Distance (WD)** — For ordinal categories on positions $\{0, 1, \ldots, K{-}1\}$, the 1-D Earth Mover's Distance uses CDFs:
$$\text{WD}_i = \sum_{j=0}^{K-2}\left|F_{\text{true}}(j) - F_{\text{pred}}(j)\right|$$
where $F_{\text{true}}(j) = \mathbb{1}[j \geq y_i]$ and $F_{\text{pred}}(j) = \sum_{k=0}^{j}\hat{p}_{ik}$.  
Averaged: $\overline{\text{WD}} = \frac{1}{N}\sum_{i} \text{WD}_i$.

This captures **how far off** the predicted distribution is.



## Why TabImpute fails

From the paper (Section A.1 - "Support for categorical variables"):
> "We support imputing categorical variables via **one-hot encodings**: First, we convert each categorical column into several one-hot encoding columns. Then, we impute missing entries within this now purely numerical matrix."
> 
> "**We leave incorporating categorical features into the pre-training procedure for future work.**"


**TabImpute treats this as a continuous numerical variable**, not categorical. When it imputes, it returns continuous values (e.g., 1.5, 2.3).



### Summary: TabPFN vs TabImpute

| Aspect | **TabPFNClassifier** | **TabImpute** |
|--------|---------------------|---------------|
| **Y Treatment** | Y is the **target** variable (separate from X) | Y is just another **column in the matrix** |
| **Y Output** | Model outputs **class probabilities** directly | Model outputs **continuous numerical values** |
| **Architecture** | Supervised learning with classification head | Unsupervised matrix completion |
| **Categorical variable** | Built-in (auto-infers categorical features) | cannot deal with catergorical variables |


### See similar X variable in both PUMA and survey, predict Y in the survey in PUMA levels
### use 2022 survey (ATP) and PUMA info (have the percentage of each district), ACS data, use this to predict 2024 election (prediction model)
### tidycensus package in R: downscaling for ACS five years data (PUMA data)